In [ ]:
# Code for classifying processed data 
# === CLASSIFICATION ===
def classify_condition(features):
    if features["settle_time"] <= 80: #eraser
        if (features["settle_time"] >= 70) and (1000 <= features["peak"] <= 2000):
            return "D:10cm, H:10cm (Coin)"
        if features["peak"] >= 2000:
            return "D:10cm, H:30cm (Eraser)"
        elif features["peak"] >= 800:
            return "D:30cm, H:30cm (Eraser)"
        else:
            return "D:10cm, H:10cm (Eraser)"
    elif (features["peak"] >= 1000) and (features["settle_time"] > 80):
        if features["peak"] >= 3200: #special case
            return "D:10cm, H:10cm (Coin)"
        elif features["peak"] >= 2500:
            return "D:30cm, H:30cm (Coin)"
        elif features["settle_time"] >= 125:
            return "D:10cm, H:30cm (Coin)"
        else: #1000<peak<2500 and 70<settle_time<140
            return "D:30cm, H:10cm (Coin)"
    else: #(peak<1000 and settle_time>80)
        return "D:10cm, H:10cm (Coin)"

def main():
    peak = 0
    peak = float(input("please enter peak observed:"))
    settle_time = 0 
    settle_time = float(input("please enter settle time observed:"))
    features = {"peak": peak,"settle_time": settle_time}
    classification = classify_condition(features)
    print("==== Feature Summary ====")
    for k, v in features.items():
        print(f"{k}: {v:.4f}")

    print(f"\nClassification: {classification}")

if __name__ == "__main__":
    main()

In [35]:
# Code for classifying softer/lighter material
# === CLASSIFICATION ===
def classify_condition(features):
    if features["peak"] <= 500:
        return "D:10cm, H:10cm (Clay)"
    elif (features["peak"] < 1200):
        return "D:30cm, H:10cm (Clay)"
    elif (features["peak"] <= 1800):
        return "D:10cm, H:30cm (Clay)"
    else:
        return "D:30cm, H:30cm (Clay)"

def main():
    peak = 0
    peak = float(input("please enter peak observed:"))
    features = {"peak": peak}
    classification = classify_condition(features)
    print("==== Feature Summary ====")
    for k, v in features.items():
        print(f"{k}: {v:.4f}")

    print(f"\nClassification: {classification}")

if __name__ == "__main__":
    main()

==== Feature Summary ====
peak: 2392.0000

Classification: D:30cm, H:30cm (Clay)


In [ ]:
#generate confusion matrix and ppv
import numpy as np
from matplotlib import pyplot as plt
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay
trueCategory = np.array([0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0])
predCategory = np.array([0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0])
#predCategory = np.array([0,0,0,0,0,1,1,0,1,1,1,1,0,0,0,1,1,1,1,1,0,1,1,1,0,1,0,0,1,1,1,0,0,1,0,1,1,1,1,0,1,1,1,1,0,0,0,0,0,0])
#predCategory = np.array([0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0])

truePos = np.sum(predCategory[trueCategory == 1] == 1)
trueNeg = np.sum(predCategory[trueCategory == 0] == 0)
predPos = np.sum(predCategory == 1)
falsePos = predPos - truePos
totalPos = np.sum(trueCategory == 1)

ppv = truePos/(truePos + falsePos)
accuracy = (truePos + trueNeg)/len(trueCategory)

cMat = confusion_matrix(trueCategory,predCategory)
disp = ConfusionMatrixDisplay(confusion_matrix=cMat, display_labels=['Not Detected', 'Detected'])
disp.plot(cmap='Blues', colorbar=True)

ax = plt.gca()
ax.set_xticklabels(['Undetected', 'Detected'])  # Add '' for alignment
ax.set_yticklabels(['Absent', 'Present'])

plt.title("Strong Performance (10N weight)")
plt.figtext(0.5, -0.05, f"Accuracy: {accuracy*100:.2f}%    PPV: {ppv:.2f}", ha='center', fontsize=12)
plt.show()

In [ ]:
# Code for data reading from STM32 
import serial
import matplotlib.pyplot as plt
import time
import datetime
import numpy as np
from scipy.fft import fft, fftfreq

# === CONFIGURATION ===
SERIAL_PORT = 'COM3'
BAUD_RATE = 115200
READ_DURATION = 5  # seconds

# === TIMESTAMPED FILENAME ===
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
PLOT_FILENAME = f"adc_plot_{timestamp}.png"

# === FEATURE EXTRACTION ===
def process_adc_data(adc_buffer, sampling_rate):
    # Time-domain features
    peak = np.max(adc_buffer)
    threshold_10 = 0.1 * peak
    threshold_90 = 0.9 * peak

    start_idx = np.argmax(adc_buffer >= threshold_10)
    end_idx = np.argmax(adc_buffer >= threshold_90)
    rise_time = (end_idx - start_idx) * (1 / sampling_rate)

    decay_start = np.argmax(adc_buffer == peak)
    decay_end = np.argmax(adc_buffer[decay_start:] < threshold_10) + decay_start
    decay_time = (decay_end - decay_start) * (1 / sampling_rate)

    energy = np.sum(np.array(adc_buffer) ** 2)

    # Frequency-domain features
    # n = len(adc_buffer)
    # yf = fft(adc_buffer - np.mean(adc_buffer))
    # xf = fftfreq(n, 1 / sampling_rate)
    # dominant_freq = abs(xf[np.argmax(np.abs(yf))])
    dominant_freq = zero_crossing_rate(adc_buffer,sampling_rate)

    #test settle time
    start_index = 0
    end_index = 0
    for i in range(len(data)):
        if data[i] == peak:
            start_index = i
            break
    for i in range(start_index,len(data)):
        if data[i] >= 10:
            end_index = i
    print(start_index,end_index)

    return {
        "peak": peak,
        "rise_time": rise_time,
        "decay_time": decay_time,
        "energy": energy,
        "dominant_freq": dominant_freq,
        "settle_time": end_index - start_index
    }

def zero_crossing_rate(signal, sampling_rate):
    """
    Estimate dominant frequency using zero-crossing method.
    
    Args:
        signal (list or np.array): Input signal (ADC values).
        sampling_rate (float): Sampling rate in Hz.

    Returns:
        estimated_freq (float): Estimated dominant frequency.
    """
    signal = np.array(signal)
    mean_val = np.mean(signal)
    centered_signal = signal - mean_val

    # Find where signal crosses zero
    zero_crossings = np.where(np.diff(np.sign(centered_signal)))[0]
    num_crossings = len(zero_crossings)

    # Each full wave has 2 zero crossings
    estimated_freq = (num_crossings / 2) * (sampling_rate / len(signal))
    return estimated_freq

# === CLASSIFICATION ===
def classify_condition(features):
    if features["settle_time"] <= 80: #eraser
        if (features["settle_time"] >= 70) and (1000 <= features["peak"] <= 2000):
            return "D:10cm, H:10cm (Coin)"
        if features["peak"] >= 2000:
            return "D:10cm, H:30cm (Eraser)"
            #2922,64
        elif features["peak"] >= 800:
            return "D:30cm, H:30cm (Eraser)"
            #1153,7
            #2512,42
            #1442,50
            #1771,34
            #1944,10
        else:
            return "D:10cm, H:10cm (Eraser)"
            #364,0
            #858,23
    elif (features["peak"] >= 1000) and (features["settle_time"] > 80):
        if features["peak"] >= 3200: #special case
            return "D:10cm, H:10cm (Coin)"
        elif features["peak"] >= 2500:
            return "D:30cm, H:30cm (Coin)"
            #1836,135
            #1170,86
            #2800,245
            #2298,173
        elif features["settle_time"] >= 125:
            return "D:10cm, H:30cm (Coin)"
        else: #1000<peak<3000 and 70<settle_time<140
            return "D:30cm, H:10cm (Coin)"
    else: #(peak<1000 and settle_time>80)
        return "D:10cm, H:10cm (Coin)"

#straw10-10
#113,327,0.8
#73,554,0.8
#466,602,0.2
#358,516,1
#63,262,1

#straw10-30
#396,617,1
#476,445,1.2
#234,463,1.4

#straw30-30
#164,553,1.6
#312,590,1.2
#107,518,1.6

#straw30-10
#88,610,0.4
#152,493,0.4
#30,532,0.8
# === CLASSIFICATION ===
def classify_condition1(features):
    if features["peak"] < 500:
        return "D:10cm, H:10cm (Clay)"
    elif features["peak"] < 1200:
        return "D:30cm, H:10cm (Clay)"
    elif features["peak"] < 1800:
        return "D:10cm, H:30cm (Clay)"
    else:
        return "D:30cm, H:30cm (Clay)"

# === INITIALIZE SERIAL ===
ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
time.sleep(2)  # Let STM32 get ready

data = []
start_time = time.time()

print("Reading data...")

while time.time() - start_time < READ_DURATION:
    line = ser.readline().decode('utf-8').strip()
    if line.isdigit():
        data.append(int(line))

ser.close()

print(f"Done reading. Saving plot as '{PLOT_FILENAME}'...")
# === ANALYZE ===
sampling_rate = len(data) / READ_DURATION  # Approximate sampling rate
features = process_adc_data(np.array(data), sampling_rate)
dominant_freq = zero_crossing_rate(np.array(data), sampling_rate)
print(f"Estimated Dominant Frequency: {dominant_freq:.2f} Hz")
classification = classify_condition1(features)

# === PLOT AND SAVE ===
# print(data)
plt.plot(data)
plt.xlabel("Sample Number")
plt.ylabel("ADC Value")
plt.title("STM32 ADC Readings 10/10/5/1 coin")
plt.grid(True)
# plt.savefig(PLOT_FILENAME)
plt.show()

# === RESULTS ===
print("==== Feature Summary ====")
for k, v in features.items():
    print(f"{k}: {v:.4f}")

print(f"\n Classification: {classification}")
if ( max(data) > 0):
    print('Item detected.\n')
else:
    print('Item not Detected\n')